# LangChain ChatOpenAI + PydanticOutputParser 예제
이 노트북은 LangChain의 `ChatOpenAI`와 `PydanticOutputParser`를 사용하여 구조화된 출력을 생성하는 방법을 보여줍니다.

- **Groq**와 **OpenAI** 두 가지 모델을 모두 사용할 수 있습니다. (Groq는 OpenAI 호환 API라서 `ChatOpenAI`로 연결)
- 설정 셀의 `PROVIDER` 값(`"groq"` 또는 `"openai"`)만 바꾸면 이후 모든 예제의 모델이 바뀝니다.
- 사용할 제공자의 키(`GROQ_API_KEY` 또는 `OPENAI_API_KEY`)가 `.env`에 있어야 합니다. OpenAI를 선택하면 사용 요금이 발생합니다.

In [ ]:
# 필수 라이브러리 설치
#uv add pydantic

In [ ]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
#load_dotenv(dotenv_path='../.env')
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 값은 출력하지 않고 설정 여부만 확인 (사용할 제공자의 키만 있으면 됨)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    print(f"{name}: {'설정됨' if key else '없음'}")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

from pydantic import BaseModel, Field
from typing import List
from pprint import pprint

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [ ]:
# 출력 구조를 정의하는 Pydantic 모델
class MovieRecommendation(BaseModel):
    movie_title: str = Field(description="추천 영화 제목")
    reason: str = Field(description="추천 이유")
    genre: List[str] = Field(description="영화 장르")
    estimated_rating: float = Field(description="10점 만점에서 예상 평점")

In [ ]:
# Pydantic 출력 파서 초기화
parser = PydanticOutputParser(pydantic_object=MovieRecommendation)
print(parser.get_format_instructions())

In [ ]:
# 프롬프트 템플릿 설정
template = """
다음 사용자 요청에 따라 영화를 추천해주세요.
요청: {query}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template)

# 파서의 지시사항을 프롬프트에 주입
prompt = prompt.partial(
    format_instructions=parser.get_format_instructions()
)
pprint(prompt)

In [ ]:
# 모델 초기화 (제공자/키/모델명은 위 공통 설정 셀의 get_llm 사용)
model = get_llm(temperature=0.5)

In [ ]:
# 체인 구성 및 실행
query = "2000년대 클래식한 느낌의 공포 영화 추천해줘"
chain = prompt | model | parser
output = chain.invoke({"query": query})

print(type(output))
pprint(output)

In [ ]:
# 결과 출력
print(f"추천 영화: {output.movie_title}")
print(f"추천 이유: {output.reason}")
print(f"장르: {', '.join(output.genre)}") # List => str 로 변환
print(f"예상 평점: {output.estimated_rating}/10")

#### PydanticOutputParser를 사용한 Review 분석 예제


In [ ]:
import pprint
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Pydantic 모델 정의 (리뷰 분석 결과 구조화)
class ReviewAnalysis(BaseModel):
    """제품 리뷰 분석 결과"""
    rating: float = Field(..., ge=0, le=5, description="5점 만점 예상 평점 (0.0 ~ 5.0)")
    pros: list[str] = Field(..., description="장점 3가지 (리스트)")
    cons: list[str] = Field(..., description="단점 3가지 (리스트)")
    summary: str = Field(..., description="리뷰 한 문장 요약")

# PydanticOutputParser 설정
parser = PydanticOutputParser(pydantic_object=ReviewAnalysis)
format_instructions = parser.get_format_instructions()

print(" 출력 형식 지시사항:")
print(format_instructions)
print("-" * 50)

# ChatPromptTemplate (부분 변수 미리 설정)
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 제품 리뷰 분석 전문가입니다. 
리뷰를 객관적으로 분석하여 다음 형식 **정확히** 따르세요:
{format_instructions}
    
- rating: 5점 만점 실수 (예: 4.2)
- pros/cons: 정확히 3개씩 리스트
- summary: 한 문장"""),
    ("user", "리뷰 내용: {review}")
]).partial(format_instructions=format_instructions)

# LLM 초기화 (재현성 위해 seed 추가, 제공자/키/모델명은 공통 설정 셀의 get_llm 사용)
model = get_llm(temperature=0.1, seed=42, provider="openai")  # temperature 를 낮춰 안정성 향상

# 체인 생성 (LCEL)
chain = prompt | model | parser

# 테스트 리뷰
review = """
이 스마트폰은 배터리 수명이 정말 좋아서 하루 종일 사용해도 충전이 필요 없었어요. 
카메라 화질도 선명하고, 특히 야간 모드가 훌륭합니다. 
다만 가격이 조금 비싸고, 무게가 200g이 넘어서 손이 피곤할 수 있어요.
"""

print(" 리뷰 분석 중...")
print(f" 리뷰: {review[:100]}...")

# 실행
try:
    output = chain.invoke({"review": review})
    
    print("\n===== 분석 결과 =====")
    print(type(output))
    pprint.pprint(output.model_dump(), width=80, indent=2)  # Pydantic 객체를 dict로 출력
    
    print(f"\n💬 요약: {output.summary}")
    
except Exception as e:
    print(f" 오류: {e}")

#### PydanticOutputParser를 사용한 날짜시간(DateTime) 예제

In [ ]:

import pprint
from datetime import datetime, timedelta
from pydantic import BaseModel, Field, field_validator
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from zoneinfo import ZoneInfo

# Pydantic 모델
class ExtractedDateTime(BaseModel):
    datetime_str: str = Field(..., description="ISO 8601")
    timezone: str = Field(default="Asia/Seoul")

    @field_validator('datetime_str')
    @classmethod
    def validate_iso(cls, v):
        try:
            datetime.fromisoformat(v.replace('Z', '+00:00'))
            return v
        except:
            raise ValueError("ISO 8601 오류")

parser = PydanticOutputParser(pydantic_object=ExtractedDateTime)

# offset-aware NOW (해결!)
NOW = datetime.now(ZoneInfo("Asia/Seoul"))  # timezone 명시!
print(f" 기준일: {NOW.strftime('%Y-%m-%d %A %H:%M:%S %Z')}")
print(f" 오늘: {NOW.strftime('%Y-%m-%d')}")
print("-" * 60)

WEEKDAYS_KO = "월화수목금토일"

def fmt_date(d: datetime) -> str:
    """날짜를 'YYYY-MM-DD (요일)' 형식의 문자열로 변환합니다."""
    return f"{d.strftime('%Y-%m-%d')} ({WEEKDAYS_KO[d.weekday()]})"

def extract_datetime_safely(text: str) -> datetime:
    """자연어 날짜 표현을 NOW 기준의 offset-aware datetime 으로 변환합니다."""
    today_str = NOW.strftime("%Y-%m-%d")
    today_weekday = NOW.strftime("%A")

    # 프롬프트 예시 날짜는 고정값이 아니라 실행일(NOW) 기준으로 계산
    in_month = NOW + timedelta(days=30)
    in_3days = NOW + timedelta(days=3)
    next_thursday = NOW - timedelta(days=NOW.weekday()) + timedelta(days=7 + 3)  # 다음 주 월요일 + 3일

    system_prompt = f"""**현재 날짜: {today_str} ({today_weekday}) 정확 사용!**

규칙:
1. "한달 후" = {today_str} + 30일 → {fmt_date(in_month)}
2. "다음 주 목요일" = {fmt_date(next_thursday)}
3. "3일 후" = {fmt_date(in_3days)}
4. 시간 없으면 09:00, Asia/Seoul(+09:00)

JSON 예시:
{{"datetime_str": "{in_month.strftime('%Y-%m-%d')}T14:00:00+09:00", "timezone": "Asia/Seoul"}}"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"텍스트: '{text}'")
    ]
    
    response = model.invoke(messages)
    parsed = parser.parse(response.content)
    
    # offset-aware 변환
    dt = datetime.fromisoformat(
        parsed.datetime_str.replace('Z', '+00:00')
    ).astimezone(ZoneInfo(parsed.timezone))
    
    return dt

# LLM (키/모델명은 공통 설정 셀의 get_llm 사용)
model = get_llm(temperature=0.0, seed=42)

# 테스트
texts = [
    "회의는 한달 후 오후 2시에 예정되어 있습니다.",
    "다음 주 목요일 마감입니다.",
    "3일 후 오후 3시30분 점검.",
    "지난주 금요일 정오에 전시회가 있었습니다."
]

print(f"\n {NOW.strftime('%Y-%m-%d')} 기준 정확 테스트")
print("=" * 70)

for i, text in enumerate(texts, 1):
    print(f"\n[{i}] '{text}'")
    
    dt = extract_datetime_safely(text)
    
    # 둘 다 offset-aware라서 빼기 가능!
    delta = dt - NOW
    print(f"  결과: {dt.strftime('%Y-%m-%d (%A) %H:%M %Z')}")
    print(f"  차이: {delta.days}일 {delta.seconds//3600}시간")
